# Study 877 — GDPNow Revisions 📉

**The Atlanta Fed's GDPNow nowcast is revised almost every day as new data lands. Is that
daily *revision* a real-time growth surprise you can trade in SPY?**

The believers' story: an **upward** revision means the incoming data beat the model's running
estimate, so stocks should rise over the next day or week; a big **downward** revision should
precede weakness. We take GDPNow's full daily forecast history (2011-08-30 → 2026-06-26,
2,102 forecasts over 60 quarters), form the within-quarter revision,
and regress forward SPY returns on it.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`6161199ecd72`); the live cells run the fast synthetic control. The nowcast posts
intraday, so the headline is the **most generous** execution (trade the release-day close).*


## 1. The idea in one line

GDPNow is a model that keeps a running guess of this quarter's GDP growth and nudges it every time a new report (jobs, ISM, retail sales) comes out. The **size of the nudge** — today's nowcast minus yesterday's — is a clean, real-time *growth surprise*. If markets hadn't already priced that surprise, an up-nudge should be followed by higher stocks. Should.

In [1]:
R = dict(b1=-3.22, t1=-1.02, r2_1=0.043, up_bps=-18.71, up_t=-2.74, down_bps=0.09, down_t=0.01)
print('predictive slope (1-day fwd SPY on the revision):')
print('  %+.2f bps per 1pp of revision   NW t = %+.2f   R2 = %.3f%%'
      % (R['b1'], R['t1'], R['r2_1']))
print('after the BIGGEST UP revisions  : %+.2f bps next day (t = %+.2f)'
      % (R['up_bps'], R['up_t']))
print('after the BIGGEST DOWN revisions: %+.2f bps next day (t = %+.2f)'
      % (R['down_bps'], R['down_t']))

predictive slope (1-day fwd SPY on the revision):
  -3.22 bps per 1pp of revision   NW t = -1.02   R2 = 0.043%
after the BIGGEST UP revisions  : -18.71 bps next day (t = -2.74)
after the BIGGEST DOWN revisions: +0.09 bps next day (t = +0.01)


## 2. What the tape says

The predictive slope is **insignificant** (NW *t* = -1.02) and the *R²* is a rounding error (**0.043%**). The one number that *is* significant goes the **wrong way**: after the biggest **up**-revisions SPY *falls* the next day (**-18.71 bps**, *t* = -2.74 — a 'sell the good news' blip), while big **down**-revisions are dead flat (+0.09 bps, *t* = +0.01). So the claim fails on both halves: up-revisions don't predict strength, and down-revisions don't precede weakness.

## 3. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: an up-revision really does lift the next-day return) and check the regression recovers it — and stays *silent* on the null (`edge=0`, revisions present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from gdpnow import data, strategy as st
null = st.synthetic_detect(data.synthetic(edge=0.0, seed=877, n=2000))
planted = st.synthetic_detect(data.synthetic(edge=0.005, seed=877, n=2000))
print('null world   : slope NW t = %+.2f  (should be ~0)' % null['t'])
print('planted world: slope NW t = %+.2f  (should light up)' % planted['t'])

null world   : slope NW t = -1.44  (should be ~0)
planted world: slope NW t = +6.65  (should light up)


## 4. The honest verdict

On 2,042 genuine daily revisions the GDPNow nowcast revision **does not predict** forward SPY: the slope is insignificant (*t* = -1.02), it **flips sign** the moment you can't trade the release-day close, and the only significant piece is *wrong-signed* and fragile. A timer that buys after up-revisions earns a Sharpe of **0.07** at 1 bp cost — versus **0.80** for just holding SPY. **Signal: None. Tradability: Mirage.** The revision is a real-time *restatement* of news the tape already absorbed minutes earlier — not a forecast of tomorrow.